# Write a Scientific Composite

A composite declares creator calls and output bindings. It has no store or scheduler access. Curators such as Filter belong in the outer pipeline, after the composition exports the data and metrics they need.

This notebook runs an inline definition with Executor. The installed equivalents used in [Pipeline Composites](../02-pipeline-design/07-composites.ipynb) can also run in fresh pipeline workers.

In [ ]:
from __future__ import annotations

from enum import StrEnum
from typing import ClassVar

from pydantic import BaseModel, Field

from artisan.composites import CompositeContext, CompositeDefinition
from artisan.execution import Executor
from artisan.operations.examples import DataGenerator, DataTransformer, MetricCalculator
from artisan.schemas import OutputSpec

In [ ]:
class GenerateTransformAndScore(CompositeDefinition):
    name = "tutorial_generate_transform_and_score"
    description = "Generate data, transform it, and export final data with metrics"

    class OutputRole(StrEnum):
        datasets = "datasets"
        metrics = "metrics"

    outputs: ClassVar[dict[str, OutputSpec]] = {
        "datasets": OutputSpec(artifact_type="csv"),
        "metrics": OutputSpec(artifact_type="metric"),
    }

    class Params(BaseModel):
        count: int = Field(default=3, ge=1, description="Number of datasets")
        seed: int = Field(default=42, description="Deterministic generation seed")
        scale_factor: float = Field(default=2.0, description="Coordinate multiplier")

    params: Params = Params()

    def compose(self, ctx: CompositeContext) -> None:
        generated = ctx.call(
            DataGenerator,
            name="generate",
            params={"count": self.params.count, "seed": self.params.seed},
        )
        transformed = ctx.call(
            DataTransformer,
            name="transform",
            inputs={"dataset": generated.output("datasets")},
            params={
                "scale_factor": self.params.scale_factor,
                "variants": 1,
                "seed": 100,
            },
        )
        scored = ctx.call(
            MetricCalculator,
            name="score",
            inputs={"dataset": transformed.output("dataset")},
        )
        ctx.output("datasets", transformed.output("dataset"))
        ctx.output("metrics", scored.output("metrics"))

`ctx.call` records wiring; it does not execute during composition. Only final datasets and metrics are exported here. The generated intermediate datasets are discarded because no final artifact requires them as a semantic referent. Scientific ancestry is not an instruction to preserve every intermediate.
Children require static input roles. ZIP and CROSS_PRODUCT grouping work inside a composition; NAME/LINEAGE grouping needs pipeline-level selection. Use `Params` to decide which children to call; execution defaults configure the outer boundary.


In [ ]:
with Executor() as executor:
    result = executor.run(GenerateTransformAndScore, params={"count": 3, "seed": 42})
    assert set(result.artifacts) == {"datasets", "metrics"}
    assert len(result.artifacts["datasets"]) == len(result.artifacts["metrics"]) == 3
    assert [edge.source_output_index for edge in result.lineage["metrics"]] == [0, 1, 2]
    assert {edge.source_role for edge in result.lineage["metrics"]} == {"datasets"}
    assert all(
        artifact.is_finalized
        for values in result.artifacts.values()
        for artifact in values
    )
assert executor.cleanup_complete
print("Three final datasets retain their three direct metrics")

In a pipeline, install this definition and submit it with `pipeline.run(...)`. It remains one outer step with nested invocation diagnostics. Submit Filter afterward, using the exported dataset stream.

A composite can declare `runner_resources`, `batch_strategy`, and `group_by` defaults, and submission overrides replace only the fields you supply. Those defaults configure the submitted boundary. When called inside another composite, it expands without reapplying its outer defaults; creator children keep their own invocation policy. See [declaring step defaults](../../how-to-guides/writing-composite-operations.md#declare-step-defaults).
